# Data Preparation Inspection

This notebook demonstrates how to inspect the data preparation logic from `run_single_super_debug`.

In [5]:
import sys
from pathlib import Path

# Add feb_exp (parent of src) and scripts so we can import src.* and data_prep_debug
feb_exp = Path("/n/home06/drooryck/codeswitching-llms/feb_exp")
sys.path.insert(0, str(feb_exp))
sys.path.insert(0, str(feb_exp / "scripts"))

from src.dataset_manager import DatasetManager
from src.model_config import ModelConfig
from data_prep_debug import prepare_data_debug

In [6]:
# Setup
prop = 0.5  # 50% French, 50% Dutch
run_id = 1
eval_prop = 0.05

data_dir = Path("/n/home06/drooryck/codeswitching-llms/feb_exp/data")
lexicon_path = Path("/n/home06/drooryck/codeswitching-llms/feb_exp/data/lexicon_sep22.json")

# Create minimal config - only required parameters needed for data preparation
# (Based on feb_exp/configs/default_model.json)
config = ModelConfig(
    # Required parameters
    n_layer=4,
    n_head=8,
    n_embd=512,
    batch_size=16,
    learning_rate=2e-4,
    warmup_steps=500,
    weight_decay=0.01,
    eval_steps=500,
    # Optional: specify training duration (one of these)
    num_train_epochs=1.0,
    # Optional: n_positions defaults to 1024 if not specified
    # vocab_size is optional and will be set from tokenizer
)

data_manager = DatasetManager(
    data_dir=str(data_dir),
    config=config,
    lexicon_path=str(lexicon_path)
)

In [7]:
# Run data preparation
train_df, val_df, test_df, train_dataloader = prepare_data_debug(
    prop=prop,
    run_id=run_id,
    eval_prop=eval_prop,
    data_manager=data_manager,
    config=config,
    batch_size=32
)

Initial train set: 591744 total | FR: 295872 | NL: 295872
Balanced train set: 591744 total | FR: 295872 | NL: 295872
Validation set: 29588 total | FR: 14794 | NL: 14794
Train pool (post-val): 562156 total | FR: 281078 | NL: 281078

Final training set:
  Total: 281078 | FR: 140539 (50.0%) | NL: 140539 (50.0%)
  Target proportion: FR=50.0%, NL=50.0%

Created DataLoader with 8784 batches
First 20 training examples (showing language interlacing):
  [  0] FR: le abeille lance le fille...
  [  1] NL: de vriend draagt de katten...
  [  2] FR: le enfant attrape le loup...
  [  3] FR: le professeur aime le aigle...
  [  4] NL: de mannen nemen de wolf...
  [  5] FR: les chevaux prennent les loups...
  [  6] NL: de beren observeren de vrouwen...
  [  7] FR: le requin casse le hibou...
  [  8] FR: les corbeaux poursuivent le ami...
  [  9] NL: de bij eet de vogels...
  [ 10] NL: de kikkers spreken de vriend...
  [ 11] FR: le ours écoute le moustique...
  [ 12] FR: les pigeons voient le souris...
 

In [8]:
# Inspect language interlacing in training set
print("Language distribution in first 100 examples:")
langs = train_df['lang'].head(100).values
print(f"FR: {(langs == 'fr').sum()}, NL: {(langs == 'nl').sum()}")
print("\nFirst 50 languages:")
print(''.join([lang.upper()[0] for lang in langs[:50]]))

Language distribution in first 100 examples:
FR: 52, NL: 48

First 50 languages:
FNFFNFNFFNNFFNNFFFFFNNNNFNNFNFNNFFFNNNNFNNFFFNFNNN


In [ ]:
# Look at consecutive language patterns
def analyze_interlacing(df, window=10):
    """Analyze how languages are interleaved."""
    langs = df['lang'].values
    
    print(f"Total examples: {len(df)}")
    print(f"FR: {(langs == 'fr').sum()}, NL: {(langs == 'nl').sum()}")
    
    # Count consecutive same-language runs
    runs = []
    current_lang = langs[0]
    run_length = 1
    
    for i in range(1, len(langs)):
        if langs[i] == current_lang:
            run_length += 1
        else:
            runs.append((current_lang, run_length))
            current_lang = langs[i]
            run_length = 1
    runs.append((current_lang, run_length))
    
    print(f"\nNumber of language runs: {len(runs)}")
    print(f"Average run length: {np.mean([r[1] for r in runs]):.2f}")
    print(f"Max run length: {max([r[1] for r in runs])}")
    
    # Show first N windows
    print(f"\nFirst {window} windows of {window} examples each:")
    for i in range(min(window, len(df) // window)):
        start = i * window
        end = start + window
        window_langs = langs[start:end]
        fr_count = (window_langs == 'fr').sum()
        nl_count = (window_langs == 'nl').sum()
        pattern = ''.join([l.upper()[0] for l in window_langs])
        print(f"  [{start:3d}-{end:3d}]: {pattern} (FR:{fr_count}, NL:{nl_count})")

analyze_interlacing(train_df)